# Expertise (VIVO): from data entry to RDF

This notebook shows how to describe a person's areas of expertise using the
[VIVO Core Ontology](https://vivoweb.org/ontology/core) and convert it into
a standardised, machine-readable RDF graph.

**You only need to edit one file:** `docs/example.oold.json`.  Everything
else is automatic.

---

## VIVO vs schema.org

Two expertise schemas are available:

| Schema | Root type | Expertise predicate | Device predicate |
|---|---|---|---|
| `expertise/schema.org/` | `foaf:Person` | `schema:knowsAbout` (all expertise) | `schema:knowsAbout` |
| `expertise/VIVO/` | `foaf:Person` | `vivo:hasResearchArea` | `vivo:hasExperienceIn` |

The VIVO schema uses **two predicates** to distinguish:
- `vivo:hasResearchArea` — scientific and engineering domains the person
  researches or publishes in (materials, modelling methods, application fields)
- `vivo:hasExperienceIn` — practical hands-on knowledge of equipment
  (measurement and production devices)

This predicate-level distinction allows SPARQL queries to retrieve
research-area expertise and device expertise separately without relying
on vocabulary-term types.

---

## What the notebook does

```
example.oold.json
  │  IRIs for each expertise area and device
  │
  ▼
RDF graph
  │  foaf:Person with vivo:hasResearchArea / vivo:hasExperienceIn
  │
  ▼
SPARQL query
  │  retrieve expertise by predicate
```

> **No transform step:** the expertise input is already in OO-LD format;
> the fields map directly to ontology IRIs.  There is nothing to pre-process.

---

## Environment setup

```bash
git clone https://github.com/Semantic-Dataspace/semantic-schemas.git
cd semantic-schemas
python3 -m venv .venv && source .venv/bin/activate
pip install semantic-schemas jupyterlab
jupyter lab
```

In [1]:
%pip install -q semantic-schemas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json, pathlib, rdflib
from semantic_schemas import Schema

HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # expertise/VIVO/

schema = Schema(SCHEMA)

---
## Step 1: Describe a person's expertise

Edit `docs/example.oold.json` with your data, then run this cell to load it.

Each field is an array of knowledge-graph IRIs:

| Field | Predicate | Description |
|---|---|---|
| `materials` | `vivo:hasResearchArea` | Materials the expert works with |
| `material_modelling` | `vivo:hasResearchArea` | Modelling / simulation approaches |
| `methods` | `vivo:hasResearchArea` | Scientific / engineering methods |
| `application_fields` | `vivo:hasResearchArea` | Application domains and industries |
| `measurement_devices` | `vivo:hasExperienceIn` | Characterisation equipment |
| `production_devices` | `vivo:hasExperienceIn` | Manufacturing equipment |

> All four research-area fields collapse to `vivo:hasResearchArea` in the RDF.
> The device-type distinction (measurement vs. production) is carried by the
> `rdf:type` of the referenced vocabulary term, not the predicate.

In [3]:
doc = json.loads((HERE / "example.oold.json").read_text())

print(json.dumps(doc, indent=2))

{
  "type": "foaf:Person",
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/expertise/VIVO/#v1.0.0",
  "materials": [
    "https://dsms.example.org/api/knowledge/mat-steel-316l",
    "https://dsms.example.org/api/knowledge/mat-alsi10mg"
  ],
  "material_modelling": [
    "https://dsms.example.org/api/knowledge/sim-fem",
    "https://dsms.example.org/api/knowledge/sim-dft"
  ],
  "methods": [
    "https://dsms.example.org/api/knowledge/method-tensile-testing",
    "https://dsms.example.org/api/knowledge/method-ebsd"
  ],
  "application_fields": [
    "https://dsms.example.org/api/knowledge/app-aerospace"
  ],
  "measurement_devices": [
    "https://dsms.example.org/api/knowledge/dev-sem",
    "https://dsms.example.org/api/knowledge/dev-xrd"
  ],
  "production_devices": [
    "https://dsms.example.org/api/knowledge/dev-lpbf"
  ]
}


---
## Step 2: Convert to RDF

The OO-LD document is parsed as JSON-LD using the ontology context from
`specs/schema.oold.yaml`.  All four research-area fields are mapped to
`vivo:hasResearchArea`; both device fields are mapped to `vivo:hasExperienceIn`.

In [4]:
flat = schema.parse(doc)

print(f"Graph contains {len(flat)} triples.\n")
print(flat.serialize(format="turtle"))

Graph contains 12 triples.



@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix vivo: <http://vivoweb.org/ontology/core#> .

[] a foaf:Person ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/expertise/VIVO/#v1.0.0> ;
    vivo:hasExperienceIn <https://dsms.example.org/api/knowledge/dev-lpbf>,
        <https://dsms.example.org/api/knowledge/dev-sem>,
        <https://dsms.example.org/api/knowledge/dev-xrd> ;
    vivo:hasResearchArea <https://dsms.example.org/api/knowledge/app-aerospace>,
        <https://dsms.example.org/api/knowledge/mat-alsi10mg>,
        <https://dsms.example.org/api/knowledge/mat-steel-316l>,
        <https://dsms.example.org/api/knowledge/method-ebsd>,
        <https://dsms.example.org/api/knowledge/method-tensile-testing>,
        <https://dsms.example.org/api/knowledge/sim-dft>,
        <https://dsms.example.org/api/knowledge/sim-fem> .




In [5]:
out_ttl = HERE / "output_expertise_vivo.ttl"
out_ttl.write_text(flat.serialize(format="turtle"))
print(f"Written to {out_ttl.name}")

Written to output_expertise_vivo.ttl


---
## Step 3: Query the graph

The SPARQL query below retrieves all expertise by predicate, making the
research-area / device distinction explicit.

In [6]:
SPARQL_VIVO = """
PREFIX vivo: <http://vivoweb.org/ontology/core#>
PREFIX foaf: <http://xmlns.com/foaf/0.1/>

SELECT ?predicate ?area
WHERE {
  ?person a foaf:Person .
  { ?person vivo:hasResearchArea ?area .
    BIND("research area" AS ?predicate) }
  UNION
  { ?person vivo:hasExperienceIn ?area .
    BIND("device experience" AS ?predicate) }
}
ORDER BY ?predicate ?area
"""

rows = list(flat.query(SPARQL_VIVO))
research = [r for r in rows if str(r.predicate) == "research area"]
devices  = [r for r in rows if str(r.predicate) == "device experience"]

print(f"Research areas ({len(research)}):")
for r in research:
    print(f"  {r.area}")

print(f"\nDevice experience ({len(devices)}):")
for r in devices:
    print(f"  {r.area}")

Research areas (7):
  https://dsms.example.org/api/knowledge/app-aerospace
  https://dsms.example.org/api/knowledge/mat-alsi10mg
  https://dsms.example.org/api/knowledge/mat-steel-316l
  https://dsms.example.org/api/knowledge/method-ebsd
  https://dsms.example.org/api/knowledge/method-tensile-testing
  https://dsms.example.org/api/knowledge/sim-dft
  https://dsms.example.org/api/knowledge/sim-fem

Device experience (3):
  https://dsms.example.org/api/knowledge/dev-lpbf
  https://dsms.example.org/api/knowledge/dev-sem
  https://dsms.example.org/api/knowledge/dev-xrd


---
## Summary

| Step | What happens |
|---|---|
| 1 | Fill in `example.oold.json` with the relevant knowledge-graph IRIs |
| 2 | The OO-LD document is parsed into an RDF graph using the VIVO context |
| 3 | SPARQL retrieves research areas and device experience by predicate |

To describe a different expert, edit `example.oold.json` and re-run all cells.

---

## Further reading

- [OO-LD primer](../../../docs/2_oold-primer.md)
- [Schema format reference](../../../docs/3_schema-format.md)
- [Expertise (schema.org)](../../schema.org/docs/1_expertise_workflow.ipynb) — alternative with a single `schema:knowsAbout` predicate